# Testing

In [1]:
from lapsim.normalisation import TransformNormalisation
import lapsim.evals.evals_suite as evals_suite
import webdataset as wd

FORESIGHT = 120
SAMPLING = 4
# NORMALISATION_BOUNDS_PATH = "bounds.json"
NORMALISATION_BOUNDS_PATH = "bounds-10.json"

DATESET = "/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-real-00.tar"

bounds = TransformNormalisation.load(NORMALISATION_BOUNDS_PATH)
bounds.transform.method = "flat-window"
bounds.transform.foresight = FORESIGHT
bounds.transform.sampling = SAMPLING


In [50]:
from lapsim.models.lapsim import LapSimModel
import webdataset
from lapsim.preprocessor.encoder import decode

model = LapSimModel(
    "ls2.pt",
    bounds
)

# evaluations = evals_suite.run_evals_suite(model, DATESET)

source_dataset = webdataset.WebDataset(DATESET)\
    .map(decode)

In [53]:
dataset = source_dataset
import json
import numpy as np
import torch

pairs = [
    
]

for record in dataset:
    normalised = bounds.normalise(record)
    transformed = bounds.transform.transform([normalised], cores=4)

    pred_pos, pred_vel = model.model(
        torch.tensor(transformed[0], dtype=torch.float32).to(model.device),
        torch.tensor(transformed[1], dtype=torch.float32).to(model.device)
    )

    pred_pos, pred_vel = model.bounds.detransform_and_denormalise(
        len(record["angles"]),
        position=pred_pos.cpu().detach().numpy(),
        velocity=pred_vel.cpu().detach().numpy()
    )

    output = {**record, "pos": pred_pos, "vel": pred_vel}
    print(list(output))
    print(output)

    pairs.append((record, output))
    
    break

['__key__', '__url__', 'acc', '__local_path__', 'angles', 'flipped', 'id', 'offsets', 'pos', 'track', 'vehicle', 'vel', 'widths']
{'__key__': 'test/Hyundai TCR - AE Dubai Autodrome - Hill Handling Circuit', '__url__': '/Users/belle/Developer/MlLapSim/dataset/spliced-again-10/lapsim-real-00.tar', 'acc': array([  3.076163  ,   2.9939332 ,   2.9058757 ,   2.8185647 ,
         2.741958  ,   2.656371  ,   2.4694571 ,   1.9525875 ,
        -0.17891084,  -5.792673  ,  -8.98941   , -10.193123  ,
       -10.852418  , -10.9776945 , -10.852781  , -10.730011  ,
       -10.537001  , -10.368977  , -10.125263  ,  -9.902446  ,
        -9.491314  ,  -8.860939  ,  -6.0381155 ,  -2.6770065 ,
         2.0818648 ,   2.137897  ,   2.2582924 ,   3.030619  ,
         3.7810912 ,   4.4018064 ,   4.766694  ,   4.7772245 ,
         3.8548841 ,  -3.0472376 ,  -4.001031  ,  -3.5731544 ,
        -3.347846  ,  -2.8171797 ,   1.017755  ,   1.6929888 ,
         1.7003365 ,   1.6530199 ,   1.6667671 ,   1.6755805 ,
   

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1230452544.py, line 6)

In [54]:
from lapsim.render import RenderItem, plot_full

for input_record, output_record in pairs:
    plot_full(
        tracks=[
            RenderItem(
                track=input_record,
                label="Predicted",
                color="red"
            ),
            RenderItem(
                track=evaluations[i][3],
                label="Ground Truth",
                color="green"
            ),
        ],
        title="..."
    )

NameError: name 'evaluations' is not defined

In [ ]:
from lapsim.eval import Evaluation
import pandas as pd

# import pandas as pd
# from neural_simulator_toolkit.lapsim.models.evaluation import Evaluation

# filter out none apex errors as they currently result in an issue
evaluatable_comparisons = [x[1] for x in evaluations if x[1].position.apex_mean is not None]

combined = Evaluation.combine(evaluatable_comparisons)

print(combined.laptime.abs_error)
print(combined.position.mean_absolute)


B## Combine Results
This combines results together based on track/vehicle results

In [ ]:
errors_by_track = {}
errors_by_vehicle = {}

for key, value, _, _ in evaluations:
    tokens = key.split(" - ")
    vehicle, track = tokens[0], " - ".join(tokens[1:])
    
    if track not in errors_by_track:
        errors_by_track[track] = []
    if vehicle not in errors_by_vehicle:
        errors_by_vehicle[vehicle] = []

    if value.position.apex_mean is not None:
        errors_by_track[track].append(value)
        errors_by_vehicle[vehicle].append(value)

for key in errors_by_vehicle:
    errors_by_vehicle[key] = Evaluation.combine(errors_by_vehicle[key])
for key in errors_by_track:
    errors_by_track[key] = Evaluation.combine(errors_by_track[key])


In [ ]:
def errors_to_df(errors):
    keys = list(errors)

    return pd.DataFrame({
        "Type": keys,
        "LapTime Error": [errors[x].laptime.abs_error for x in keys],
        "LapTime Percentage": [errors[x].laptime.percentage for x in keys],
        "LapTime Error Per Minute": [errors[x].laptime.error_per_minute for x in keys],

        "Position Error": [errors[x].position.mean_absolute for x in keys],
        "Position ci95": [errors[x].position.ci95 for x in keys],

        "Velocity Error": [errors[x].velocity.mean_absolute for x in keys],
        "Velocity ci95": [errors[x].velocity.ci95 for x in keys]
    })

vehicles_df = errors_to_df(errors_by_vehicle)
track_df = errors_to_df(errors_by_track)


In [ ]:
track_df